<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/copy_Notebook_1_GEE_Forest_Area_Computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 1 — GEE Forest Area Computation
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [2]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

In [5]:
## ── CELL 2: conect to drive  ─────────────────────────────────────────────────
# Drive paths — for GEE raw exports
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
GEE_FOLDER = '/content/drive/MyDrive/BiocharProject/GEE_exports/'
FAO_FOLDER   = '/content/drive/MyDrive/BiocharProject/FAO_data/'

from google.colab import drive
drive.mount('/content/drive')

print('✅ connected to google drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ connected to google drive


In [6]:
# ── CELL 2: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main

DATA_folder = '/content/Biochar_forest_estimation/data/'

print('✅ Ready')

/content/Biochar_forest_estimation
HEAD is now at 3e43409 Refactor region country lists in data_config.py
✅ Ready


In [7]:
# ── CELL 3: import libraries and data from data_config ─────────────────────

import ee
import geemap
import pandas as pd
from data_config import (FAO_LSIB_REGION, us_state_names,
                         get_all_countries, build_country_lookup,
                         country_thresholds, state_thresholds

                        )

print('✅ Libraries imported')
print('✅ Data config loaded')

✅ Libraries imported
✅ Data config loaded


In [8]:
# ── CELL 4: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')


✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded


In [9]:
# ── CELL 5: processing datasets ───────────────────────────────────────

# Select & Mask Hansen Bands
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

print('✅ GLC FCS30D 2000 loaded')
print('✅ GLC forest mask created (classes 51-92)')

# Forest Class Definitions
forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'✅ {len(forestClasses)} forest classes defined')

✅ treecover2000 masked
✅ GLC FCS30D 2000 loaded
✅ GLC forest mask created (classes 51-92)
✅ 10 forest classes defined


In [11]:
# ── CELL 6: GEE Functions — Countries ────────────────────────────────────────
def prepare_forest_collection(selected_regions, threshold):
    """
    Prepare a GEE FeatureCollection with forest area per country
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    all_countries = get_all_countries(selected_regions)
    lsib_fao = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
                 .filter(ee.Filter.inList('country_na', all_countries))
    forest_mask = treecover2000_masked.gte(threshold).selfMask().updateMask(datamask.eq(1))
    area_image  = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))
    region_areas = area_image.reduceRegions(
        collection=lsib_fao,
        reducer=ee.Reducer.sum(),
        scale=30,
    )
    region_areas = region_areas.map(lambda f: f.set('threshold', threshold))
    return region_areas


def export_forest_area(selected_regions, thresholds):
    """Submit one export task per threshold to Google Drive."""
    tasks = []
    for threshold in thresholds:
        fc = prepare_forest_collection(selected_regions, threshold)
        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'forest_area_{threshold}',
            folder='GEE_exports',
            fileNamePrefix=f'forest_area_{threshold}',
            fileFormat='CSV',
            selectors=['country_na', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'✅ Task submitted for threshold {threshold}%')
    return tasks

print('✅ prepare_forest_collection() defined')
print('✅ export_forest_area() defined')

✅ prepare_forest_collection() defined
✅ export_forest_area() defined


In [23]:
# ── CELL 7: GEE Functions — US States ────────────────────────────────────────
def prepare_states_forest_collection(selected_states, threshold):
    """
    Prepare a GEE FeatureCollection with forest area per US state
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    tiger_state_names =  ee.FeatureCollection('TIGER/2018/States').filter(ee.Filter.inList('NAME', us_state_names))

    forest_mask = treecover2000_masked.gte(threshold).selfMask().updateMask(datamask.eq(1))
    area_image  = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))
    states_areas = area_image.reduceRegions(
        collection=tiger_state_names,
        reducer=ee.Reducer.sum(),
        scale=30,
        maxPixelsPerRegion=1e13
    )
    states_areas = states_areas.map(lambda f: f.set('threshold', threshold))
    return states_areas


def export_states_forest_area(selected_states, thresholds):
    """Submit one export task per threshold to Google Drive."""
    tasks = []
    for threshold in thresholds:
        fc = prepare_states_forest_collection(selected_states, threshold)
        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'states_forest_area_{threshold}',
            folder='GEE_exports',
            fileNamePrefix=f'states_forest_area_{threshold}',
            fileFormat='CSV',
            selectors=['NAME', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'✅ Task submitted for threshold {threshold}%')
    return tasks

print('✅ prepare_states_forest_collection() defined')
print('✅ export_states_forest_area() defined')

✅ prepare_states_forest_collection() defined
✅ export_states_forest_area() defined


In [21]:
# ── CELL 8: Run Exports ───────────────────────────────────────────────────────
print('── Submitting country tasks ──')
country_tasks = export_forest_area(FAO_LSIB_REGION, country_thresholds)

print('\n── Submitting state tasks ──')
state_tasks = export_states_forest_area(us_state_names, state_thresholds)

── Submitting country tasks ──
✅ Task submitted for threshold 10%
✅ Task submitted for threshold 20%
✅ Task submitted for threshold 30%
✅ Task submitted for threshold 40%
✅ Task submitted for threshold 50%

── Submitting state tasks ──
✅ Task submitted for threshold 10%
✅ Task submitted for threshold 20%
✅ Task submitted for threshold 30%
✅ Task submitted for threshold 40%
✅ Task submitted for threshold 50%


In [24]:
# ── CELL 9: Monitor Tasks ─────────────────────────────────────────────────────
import time

all_tasks = country_tasks + state_tasks

for i in range(35):
    print(f'\n── Check {i+1} ──')
    all_done = True
    for task in all_tasks:
        status = task.status()
        print(f"  {status['description']}: {status['state']}")
        if status['state'] not in ['COMPLETED', 'FAILED']:
            all_done = False
    if all_done:
        print('\n✅ All tasks completed!')
        break
    time.sleep(60)


── Check 1 ──
  forest_area_10: COMPLETED
  forest_area_20: COMPLETED
  forest_area_30: COMPLETED
  forest_area_40: COMPLETED
  forest_area_50: COMPLETED
  states_forest_area_10: COMPLETED
  states_forest_area_20: COMPLETED
  states_forest_area_30: COMPLETED
  states_forest_area_40: COMPLETED
  states_forest_area_50: COMPLETED

✅ All tasks completed!


In [27]:
# ── CELL 10: Copy GEE exports from Drive to GitHub repo ──────────────────────
import shutil

GEE_FOLDER = '/content/drive/MyDrive/GEE_exports/'
DATA_FOLDER = '/content/Biochar_forest_estimation/data/'

files = [
    f'forest_area_{t}.csv'        for t in country_thresholds
] + [
    f'states_forest_area_{t}.csv' for t in state_thresholds
]

for f in files:
    df=pd.read_csv(GEE_FOLDER + f)

    if 'country_na' in df.columns:
      df=df.groupby('country_na').sum().reset_index()
      df=df.rename(columns={'country_na': 'country', 'sum' : 'area_Mha'})

    elif 'NAME' in df.columns:
      df=df.groupby('NAME').sum().reset_index()
      df=df.rename(columns={'NAME': 'state', 'sum' : 'area_Mha'})

    df.to_csv(DATA_FOLDER + f, index=False)

    shutil.copy(GEE_FOLDER + f, DATA_FOLDER + f)
    print(f'✅ Copied {f}')


✅ Copied forest_area_10.csv
✅ Copied forest_area_20.csv
✅ Copied forest_area_30.csv
✅ Copied forest_area_40.csv
✅ Copied forest_area_50.csv
✅ Copied states_forest_area_10.csv
✅ Copied states_forest_area_20.csv
✅ Copied states_forest_area_30.csv
✅ Copied states_forest_area_40.csv
✅ Copied states_forest_area_50.csv


In [28]:
# ── CELL 11: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in dir():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)

Enter PAT: ··········
/content/Biochar_forest_estimation
[main 1285463] update results
 11 files changed, 1360 insertions(+), 1360 deletions(-)
 rewrite data/forest_area_10.csv (64%)
 rewrite data/forest_area_20.csv (66%)
 rewrite data/forest_area_30.csv (67%)
 rewrite data/forest_area_40.csv (67%)
 rewrite data/forest_area_50.csv (62%)
 rewrite data/states_forest_area_10.csv (96%)
 rewrite data/states_forest_area_20.csv (92%)
 rewrite data/states_forest_area_30.csv (96%)
 rewrite data/states_forest_area_40.csv (88%)
 rewrite data/states_forest_area_50.csv (86%)
✅ Pushed to GitHub
